# ResNet50 End-to-End Transfer Learning

This notebook implements ResNet50 transfer learning for four-class brain tumour MRI classification using the predefined five-fold cross-validation assignments.

In [1]:
import sys
import tensorflow as tf

print("Python executable:")
print(sys.executable)

print("\nPython version:")
print(sys.version)

print("\nTensorFlow version:")
print(tf.__version__)

print("\nAvailable GPUs:")
print(tf.config.list_physical_devices("GPU"))

Python executable:
c:\Users\kwsta\venvs\resnet50\Scripts\python.exe

Python version:
3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]

TensorFlow version:
2.21.0

Available GPUs:
[]


In [ ]:
# Import TensorFlow and use the shorter name "tf".
import tensorflow as tf

# Import the prebuilt ResNet50 architecture from TensorFlow/Keras.
from tensorflow.keras.applications import ResNet50


# Print the installed TensorFlow version.
print("TensorFlow version:", tf.__version__)

# Print a progress message before loading the model.
print("Starting ResNet50 loading...")


try:
    # Create the ResNet50 base model.
    resnet50_base = ResNet50(
        # Load weights that were previously trained on the ImageNet dataset.
        weights="imagenet",

        # Remove ResNet50's original classifier, which predicts
        # the 1,000 ImageNet classes.
        # We keep only the feature-extraction part of the network.
        include_top=False,

        # ResNet50 expects images that are:
        # 224 pixels high, 224 pixels wide, with 3 RGB channels.
        input_shape=(224, 224, 3),

        # Apply global average pooling to the final feature maps.
        # This produces one 2,048-value feature vector per image.
        pooling="avg"
    )

    # Freeze the complete pretrained ResNet50 model.
    # Its ImageNet weights will not be changed during training.
    # The model will be used only as a fixed feature extractor.
    resnet50_base.trainable = False

    print("\nResNet50 loaded successfully.")

    # Print the internal name assigned to the model.
    print("Model name:", resnet50_base.name)

    # Print the expected input shape.
    # None represents the batch size, which can vary.
    print("Input shape:", resnet50_base.input_shape)

    # Print the output shape.
    # With pooling="avg", each image produces 2,048 features.
    print("Output shape:", resnet50_base.output_shape)

    # Count all individual numerical parameters in the model.
    # This includes both trainable and non-trainable parameters.
    print(
        "Total parameters:",
        resnet50_base.count_params()
    )

    # Count the variable tensors that can be updated during training.
    # This should be 0 because the complete model was frozen.
    print(
        "Trainable variables:",
        len(resnet50_base.trainable_variables)
    )

    # Count the variable tensors that remain fixed.
    # One variable may contain thousands or millions of parameters.
    print(
        "Non-trainable variables:",
        len(resnet50_base.non_trainable_variables)
    )


except Exception as error:
    # Run this section if ResNet50 cannot be loaded.
    # Possible causes include download, memory, or configuration problems.
    print("\nResNet50 loading failed.")

    # Print the name of the error type.
    print("Error type:", type(error).__name__)

    # Print the detailed error message.
    print("Error message:", error)

TensorFlow version: 2.21.0
Starting ResNet50 loading...
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 15s 0us/step

ResNet50 loaded successfully.
Model name: resnet50
Input shape: (None, 224, 224, 3)
Output shape: (None, 2048)
Total parameters: 23587712
Trainable variables: 0
Non-trainable variables: 318


In [4]:
# Reproducible ResNet50 experiment setup

import os
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf


# ---------------------------------------------------------
# 1. Reproducibility
# ---------------------------------------------------------

RANDOM_SEED = 42

os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)


# Request deterministic TensorFlow operations where supported
try:
    tf.config.experimental.enable_op_determinism()
    determinism_enabled = True
except Exception as error:
    determinism_enabled = False
    determinism_error = str(error)


# ---------------------------------------------------------
# 2. Project paths
# ---------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

DATA_DIR = (
    PROJECT_ROOT
    / "processed_data_cropped"
)

FOLDS_FILE = (
    PROJECT_ROOT
    / "splits"
    / "five_fold_cross_validation.csv"
)

RESULTS_DIR = (
    PROJECT_ROOT
    / "results"
    / "resnet50_transfer_learning"
)


# ---------------------------------------------------------
# 3. Initial experiment configuration
# ---------------------------------------------------------

IMAGE_HEIGHT = 224
IMAGE_WIDTH = 224
IMAGE_CHANNELS = 3

NUMBER_OF_CLASSES = 4

BATCH_SIZE = 16

CLASS_NAMES = [
    "glioma",
    "meningioma",
    "notumor",
    "pituitary"
]


# ---------------------------------------------------------
# 4. Display configuration
# ---------------------------------------------------------

print("--- Environment ---")
print("Python version:", sys.version)
print("Python executable:", sys.executable)
print("TensorFlow version:", tf.__version__)
print("Keras version:", tf.keras.__version__)

print("\n--- Reproducibility ---")
print("Random seed:", RANDOM_SEED)
print("Deterministic operations enabled:", determinism_enabled)

if not determinism_enabled:
    print("Determinism error:", determinism_error)

print("\n--- Project paths ---")
print("Project root:", PROJECT_ROOT)
print("Dataset folder:", DATA_DIR)
print("Dataset folder exists:", DATA_DIR.exists())
print("Fold file:", FOLDS_FILE)
print("Fold file exists:", FOLDS_FILE.exists())

print("\n--- Experiment configuration ---")
print(
    "Input shape:",
    (
        IMAGE_HEIGHT,
        IMAGE_WIDTH,
        IMAGE_CHANNELS
    )
)
print("Number of classes:", NUMBER_OF_CLASSES)
print("Batch size:", BATCH_SIZE)
print("Class names:", CLASS_NAMES)

--- Environment ---
Python version: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
Python executable: c:\Users\kwsta\venvs\resnet50\Scripts\python.exe
TensorFlow version: 2.21.0
Keras version: 3.15.1

--- Reproducibility ---
Random seed: 42
Deterministic operations enabled: True

--- Project paths ---
Project root: c:\Users\kwsta\OneDrive\Desktop\Desetation project\brain-tumour-mri-classification
Dataset folder: c:\Users\kwsta\OneDrive\Desktop\Desetation project\brain-tumour-mri-classification\processed_data_cropped
Dataset folder exists: True
Fold file: c:\Users\kwsta\OneDrive\Desktop\Desetation project\brain-tumour-mri-classification\splits\five_fold_cross_validation.csv
Fold file exists: True

--- Experiment configuration ---
Input shape: (224, 224, 3)
Number of classes: 4
Batch size: 16
Class names: ['glioma', 'meningioma', 'notumor', 'pituitary']
